# 03 — Fine-tune Qwen3-1.7B with LoRA in NeMo

This notebook demonstrates a config-driven NeMo 2.0 launch pattern. Reading the config, validating paths, and previewing the launch are laptop-safe. Building the NeMo recipe requires the full NeMo environment. Actual training is disabled and requires a compatible Linux/CUDA host with an NVIDIA GPU.

The first real run should remain a two-step smoke test. The sample dataset is intentionally too small for meaningful model quality.

In [1]:
from pathlib import Path
import json
import os
import shutil

import yaml

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / 'configs' / 'qwen_lora_config.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

print(yaml.safe_dump(config, sort_keys=False))

model:
  name: Qwen/Qwen3-1.7B
  local_alternative: Qwen/Qwen2.5-1.5B-Instruct
data:
  train_path: data/train.jsonl
  validation_path: data/validation.jsonl
  test_path: data/test.jsonl
  max_sequence_length: 512
output:
  path: outputs/qwen3_1p7b_lora
lora:
  rank: 16
  alpha: 32
  dropout: 0.05
  target_modules:
  - linear_qkv
  - linear_proj
  - linear_fc1
  - linear_fc2
training:
  learning_rate: 0.0002
  batch_size: 1
  global_batch_size: 8
  epochs: 1
  precision: bf16-mixed
  num_gpus: 1
  seed: 42
smoke_test:
  enabled: true
  max_steps: 2



In [2]:
required_sections = {'model', 'data', 'output', 'lora', 'training', 'smoke_test'}
missing_sections = required_sections - set(config)
assert not missing_sections, f'Missing config sections: {sorted(missing_sections)}'

for key in ['train_path', 'validation_path', 'test_path']:
    path = PROJECT_ROOT / config['data'][key]
    assert path.is_file(), f'Missing data file: {path}'

assert config['model']['name'] == 'Qwen/Qwen3-1.7B'
assert config['lora']['rank'] > 0
assert config['training']['batch_size'] > 0
assert config['training']['global_batch_size'] >= config['training']['batch_size']
print('Configuration and source paths passed laptop-safe checks.')

Configuration and source paths passed laptop-safe checks.


## Stage data for NeMo's `FineTuningDataModule`

NeMo's basic fine-tuning data module accepts `input`/`output` JSONL and expects files named `training.jsonl`, `validation.jsonl`, and `test.jsonl` in one dataset directory. The helper below converts chat records to that minimum representation in an ignored `outputs/nemo_data/` directory. Source data is never modified.

For a production chat-tuning pipeline, use the NeMo chat data module and chat template supported by the exact NeMo release rather than flattening multi-turn conversations.

In [3]:
PREPARE_NEMO_DATA = True  # Set True before building the real recipe.
NEMO_DATA_ROOT = PROJECT_ROOT / 'outputs' / 'nemo_data'

def load_jsonl(path: Path) -> list[dict]:
    with path.open('r', encoding='utf-8-sig') as handle:
        return [json.loads(line) for line in handle if line.strip()]

def to_input_output(record: dict) -> dict[str, str]:
    if 'input' in record:
        return {'input': record['input'].strip(), 'output': record['output'].strip()}
    messages = record['messages']
    assistant_indices = [i for i, message in enumerate(messages) if message['role'] == 'assistant']
    if not assistant_indices:
        raise ValueError('Chat record has no assistant response.')
    answer_index = assistant_indices[-1]
    prompt_messages = messages[:answer_index]
    prompt = '\n'.join(
        f"{message['role'].capitalize()}: {message['content'].strip()}"
        for message in prompt_messages
    )
    return {'input': prompt, 'output': messages[answer_index]['content'].strip()}

source_to_target = {
    'train_path': 'training.jsonl',
    'validation_path': 'validation.jsonl',
    'test_path': 'test.jsonl',
}

if PREPARE_NEMO_DATA:
    NEMO_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for source_key, target_name in source_to_target.items():
        source_path = PROJECT_ROOT / config['data'][source_key]
        target_path = NEMO_DATA_ROOT / target_name
        records = [to_input_output(record) for record in load_jsonl(source_path)]
        with target_path.open('w', encoding='utf-8', newline='\n') as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        print(f'Wrote {len(records)} records to {target_path}')
else:
    print('Data staging skipped. Set PREPARE_NEMO_DATA=True on the GPU host.')

Wrote 8 records to C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\outputs\nemo_data\training.jsonl
Wrote 3 records to C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\outputs\nemo_data\validation.jsonl
Wrote 3 records to C:\Code\finetuning\nvftqwen\nemo-qwen-hands-on-series\outputs\nemo_data\test.jsonl


In [1]:
from nemo.collections import llm

ModuleNotFoundError: No module named 'nemo'

## Build the NeMo recipe

The current NeMo Qwen3 recipe family includes `qwen3_1p7b.finetune_recipe`. LoRA is applied through the recipe's PEFT object. Rank is named `dim` in NeMo. The recipe remains version-sensitive, so inspect the object on the target NeMo container before launching.

The code maps the YAML's data paths, output path, rank, alpha, dropout, target modules, learning rate, batch sizes, epochs, and smoke-test limit. The recipe supplies its release-tested mixed-precision plugin; the YAML precision value is checked and printed so it can be compared to that recipe before execution.

In [4]:
BUILD_NEMO_RECIPE = True  # Requires requirements.txt on a compatible Linux/CUDA host.
recipe = None

if BUILD_NEMO_RECIPE:
    if not (NEMO_DATA_ROOT / 'training.jsonl').is_file():
        raise RuntimeError('Stage the NeMo data first by setting PREPARE_NEMO_DATA=True.')

    from nemo.collections import llm

    qwen_recipe_group = getattr(llm, 'qwen3_1p7b', None)
    if qwen_recipe_group is None or not hasattr(qwen_recipe_group, 'finetune_recipe'):
        raise RuntimeError(
            'This NeMo release does not expose llm.qwen3_1p7b.finetune_recipe. '
            'Use a current NeMo 2.0 release or container and review its Qwen3 recipe names.'
        )

    output_dir = PROJECT_ROOT / config['output']['path']
    recipe = qwen_recipe_group.finetune_recipe(
        name='qwen3_1p7b_lora',
        dir=str(output_dir),
        num_nodes=1,
        num_gpus_per_node=config['training']['num_gpus'],
        peft_scheme='lora',
        packed_sequence=False,
    )
    recipe.data = llm.FineTuningDataModule(
        dataset_root=str(NEMO_DATA_ROOT),
        seq_length=config['data']['max_sequence_length'],
        micro_batch_size=config['training']['batch_size'],
        global_batch_size=config['training']['global_batch_size'],
        num_workers=0,
    )
    recipe.peft.target_modules = config['lora']['target_modules']
    recipe.peft.dim = config['lora']['rank']
    recipe.peft.alpha = config['lora']['alpha']
    recipe.peft.dropout = config['lora']['dropout']
    recipe.optim.config.lr = float(config['training']['learning_rate'])
    recipe.trainer.max_epochs = config['training']['epochs']
    if config['smoke_test']['enabled']:
        recipe.trainer.max_steps = config['smoke_test']['max_steps']

    print('Recipe built:', recipe)
    print('Requested precision:', config['training']['precision'])
    print('Inspect recipe.trainer.plugins before launch to verify actual precision.')
else:
    print('Recipe build skipped. YAML validation above is the laptop-safe dry run.')

ModuleNotFoundError: No module named 'nemo'

## GPU launch gate

Before enabling training:

1. Run `python scripts/check_gpu.py` on the target host.
2. Keep `smoke_test.enabled: true` and `max_steps: 2`.
3. Confirm the output directory has enough space and contains no work you need to preserve.
4. Confirm the NeMo recipe's checkpoint import/resume configuration points to `hf://Qwen/Qwen3-1.7B`.
5. Start with one GPU, sequence length 512, and micro batch size 1. Reduce sequence length if the first step runs out of memory.

`direct=True` is used for notebook execution. For scheduled or multi-process jobs, move the recipe to a Python entry point with an `if __name__ == '__main__':` guard and select an appropriate NeMo-Run executor.

In [7]:
RUN_TRAINING = False  # Deliberate safety gate: CUDA/GPU required.

if RUN_TRAINING:
    if os.name == 'nt':
        raise RuntimeError('Run NeMo LLM training in a supported Linux/CUDA environment.')
    import torch
    import nemo_run as run

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available in this kernel.')
    if recipe is None:
        raise RuntimeError('Set BUILD_NEMO_RECIPE=True and run the recipe cell first.')
    print('Starting training with:', torch.cuda.get_device_name(0))
    run.run(recipe, direct=True)
else:
    print('Training skipped. Set RUN_TRAINING=True only on the prepared GPU host.')

Training skipped. Set RUN_TRAINING=True only on the prepared GPU host.


## After the smoke test

Verify that two optimizer steps completed, loss was finite, a checkpoint/log directory was created, and GPU memory stayed within budget. Only then disable the smoke-test limit and rerun with a real dataset. Record the exact container, NeMo, driver, CUDA, model revision, config, and data revision for reproducibility.